In [ ]:
"""
Script 6: Correlation Analysis
Analyze correlations between variables using Pearson and Spearman methods
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def calculate_correlations(df):
    """Calculate Pearson and Spearman correlations"""
    numeric_df = df.select_dtypes(include=[np.number]).drop('No', axis=1, errors='ignore')
    
    # Pearson correlation
    pearson_corr = numeric_df.corr(method='pearson')
    
    # Spearman correlation
    spearman_corr = numeric_df.corr(method='spearman')
    
    return numeric_df, pearson_corr, spearman_corr

def print_correlation_matrix(corr_matrix, method_name):
    """Print correlation matrix"""
    print(f"\n{'='*70}")
    print(f"{method_name.upper()} CORRELATION MATRIX")
    print(f"{'='*70}\n")
    print(corr_matrix.to_string())

def perform_correlation_tests(numeric_df):
    """Perform individual correlation tests"""
    print(f"\n{'='*70}")
    print("PAIRWISE CORRELATION TESTS")
    print(f"{'='*70}\n")
    
    columns = numeric_df.columns
    results = []
    
    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            col1 = columns[i]
            col2 = columns[j]
            
            # Get data without NaN
            data1 = numeric_df[col1].dropna()
            data2 = numeric_df[col2].dropna()
            
            # Find common indices
            common_idx = data1.index.intersection(data2.index)
            if len(common_idx) < 2:
                continue
            
            x = numeric_df.loc[common_idx, col1]
            y = numeric_df.loc[common_idx, col2]
            
            # Pearson correlation
            pearson_r, pearson_p = scipy_stats.pearsonr(x, y)
            
            # Spearman correlation
            spearman_r, spearman_p = scipy_stats.spearmanr(x, y)
            
            result = {
                'Variable 1': col1,
                'Variable 2': col2,
                'Pearson r': pearson_r,
                'Pearson p-value': pearson_p,
                'Pearson Significant': 'Yes' if pearson_p < 0.05 else 'No',
                'Spearman r': spearman_r,
                'Spearman p-value': spearman_p,
                'Spearman Significant': 'Yes' if spearman_p < 0.05 else 'No'
            }
            
            results.append(result)
            
            print(f"{col1} vs {col2}")
            print("─" * 70)
            print(f"  Pearson r: {pearson_r:>10.6f}  |  p-value: {pearson_p:>12.6e}  |  Significant: {result['Pearson Significant']}")
            print(f"  Spearman r: {spearman_r:>9.6f}  |  p-value: {spearman_p:>12.6e}  |  Significant: {result['Spearman Significant']}")
            print()
    
    return pd.DataFrame(results)

def interpret_correlation(r_value):
    """Interpret correlation strength"""
    abs_r = abs(r_value)
    if abs_r < 0.3:
        return "Weak"
    elif abs_r < 0.7:
        return "Moderate"
    else:
        return "Strong"

def plot_correlation_heatmap(corr_matrix, method_name):
    """Plot correlation heatmap"""
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    
    plt.title(f'{method_name} Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    file_name = f'outputs/plots/{method_name.lower().replace(" ", "_")}_heatmap.png'
    Path('outputs/plots').mkdir(parents=True, exist_ok=True)
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {file_name}")
    plt.close()

def correlation_summary(numeric_df, corr_matrix):
    """Generate correlation summary"""
    print(f"\n{'='*70}")
    print("CORRELATION STRENGTH ANALYSIS")
    print(f"{'='*70}\n")
    
    columns = numeric_df.columns
    
    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            col1 = columns[i]
            col2 = columns[j]
            
            r_value = corr_matrix.loc[col1, col2]
            strength = interpret_correlation(r_value)
            
            if r_value > 0:
                direction = "Positive"
            elif r_value < 0:
                direction = "Negative"
            else:
                direction = "No"
            
            print(f"{col1} ↔ {col2}")
            print(f"  Correlation: {r_value:>8.4f}  |  Strength: {strength:>10}  |  Direction: {direction}")
            print()

if __name__ == "__main__":
    file_path = 'outputs/cleaned_data.csv'
    
    try:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        
        # Calculate correlations
        numeric_df, pearson_corr, spearman_corr = calculate_correlations(df)
        
        # Print correlation matrices
        print_correlation_matrix(pearson_corr, "Pearson")
        print_correlation_matrix(spearman_corr, "Spearman")
        
        # Perform pairwise tests
        test_results = perform_correlation_tests(numeric_df)
        
        # Print correlation summary
        correlation_summary(numeric_df, pearson_corr)
        
        # Plot heatmaps
        plot_correlation_heatmap(pearson_corr, "Pearson")
        plot_correlation_heatmap(spearman_corr, "Spearman")
        
        # Save results
        Path('outputs').mkdir(exist_ok=True)
        pearson_corr.to_csv('outputs/pearson_correlation.csv')
        spearman_corr.to_csv('outputs/spearman_correlation.csv')
        test_results.to_csv('outputs/correlation_tests.csv', index=False)
        
        print(f"\n✓ Correlation results saved to outputs/")
        print("✓ Correlation analysis completed!")
        
    except FileNotFoundError:
        print(f"Please run 01_data_loading.py first to generate {file_path}")